# Tests d'indépendance

## Idée: Y= f(X) + epsilon
### Tester dans les deux sens et étudier les indépendances avec L1, L2, HSIC, HSIC simplifié, matrices de précision, test multiple, information mutuelle

In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import mutual_info_regression
from scipy.stats import entropy

In [ ]:
n = 1000
d = {}
d["X"] = np.random.uniform(-1, 1, n) 
d["Y"] = d["X"] * 1.5 + np.random.uniform(-1, 1, n)  
df = pd.DataFrame(d)

X = pd.DataFrame(df["X"])
Y = pd.DataFrame(df["Y"])

lr_dir = LinearRegression().fit(df[['X']], df['Y'])
res_dir = (df['Y'] - lr_dir.predict(df[['X']])).values

lr_inv = LinearRegression().fit(df[['Y']], df['X'])
res_inv = (df['X'] - lr_inv.predict(df[['Y']])).values


In [ ]:
#test correlation 



def test_suite(cause, residual):
    cause_flat = cause.values.flatten()
    res_flat = residual.flatten()
    res_flat = (res_flat - np.mean(res_flat)) / np.std(res_flat)
    
    # hsic appoximé
    hsic_proxy = abs(np.corrcoef(cause_flat, res_flat**2)[0,1])
    
    # info mutuelle
    mi = mutual_info_regression(cause_flat.reshape(-1, 1), res_flat)[0]
    
    # matrice précision
    cov = np.cov(cause_flat, res_flat)
    precision = np.linalg.inv(cov)[0,1]
    
    # kl div 
    def kl_proxy(x):
        hist, bin_edges = np.histogram(x, bins=30, density=True)
        return entropy(hist + 1e-10) 
    
    kl = kl_proxy(res_flat)

    return {"HSIC_proxy": hsic_proxy, "MI": mi, "Precision": abs(precision), "Entropy": kl}


print("X->Y")
print(test_suite(df['X'], res_dir))

print("Y->X")
print(test_suite(df['Y'], res_inv))

In [ ]:
def plot_causal_diagnostics(df, x_col, y_col):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    mod1 = LinearRegression().fit(df[[x_col]], df[y_col])
    res1 = df[y_col] - mod1.predict(df[[x_col]])
    axes[0].scatter(df[x_col], res1, alpha=0.5, c='green')
    axes[0].set_title(f"direct: {x_col} -> {y_col}")
    axes[0].axhline(0, color='black', linestyle='--')
    
    mod2 = LinearRegression().fit(df[[y_col]], df[x_col])
    res2 = df[x_col] - mod2.predict(df[[y_col]])
    axes[1].scatter(df[y_col], res2, alpha=0.5, c='purple')
    axes[1].set_title(f"inverse: {y_col} -> {x_col}")
    axes[1].axhline(0, color='black', linestyle='--')
    
    plt.tight_layout()
    plt.show()
    



plot_causal_diagnostics(df, "X","Y")

In [ ]:
n = 1000
d = {}
d["X"] = np.random.uniform(-3, 3, n) 
d["Y"] = d["X"] * 3 + np.random.uniform(-2, 2, n)  
df = pd.DataFrame(d)

X = pd.DataFrame(df["X"])
Y = pd.DataFrame(df["Y"])

lr_dir = LinearRegression().fit(df[['X']], df['Y'])
res_dir = (df['Y'] - lr_dir.predict(df[['X']])).values

lr_inv = LinearRegression().fit(df[['Y']], df['X'])
res_inv = (df['X'] - lr_inv.predict(df[['Y']])).values


print("X->Y")
print(test_suite(df['X'], res_dir))

print("Y->X")
print(test_suite(df['Y'], res_inv))

plot_causal_diagnostics(df, "X","Y")